In [188]:
import csv
from dotenv import load_dotenv
import os
from langchain_neo4j import Neo4jGraph
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

In [189]:
load_dotenv(override=True)

True

In [190]:
AURA_INSTANCENAME = os.environ["AURA_INSTANCENAME"]
NEO4J_URI = os.environ["NEO4J_URI"]
NEO4J_USERNAME = os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]
AUTH = (NEO4J_USERNAME, NEO4J_PASSWORD)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_ENDPOINT = os.getenv("OPENAI_ENDPOINT")

In [191]:
chat = ChatOpenAI(api_key=OPENAI_API_KEY, temperature=0, model="gpt-4o-mini")

In [192]:
kg = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
) 

In [193]:
# Delete existing data in the graph database.
def delete_existing_data():
    kg.query("""
    MATCH (n)
    DETACH DELETE n
    """)

In [194]:
def create_node(node_type: str, name: str, description: str = ""):
    query = f"""
    MERGE (n:Entity:{node_type} {{name: $name}})
    SET n.description = $description,
        n.updatedOn = datetime()
    RETURN elementId(n) AS nodeId,
           labels(n) AS labels,
           n.name AS name,
           n.description AS description
    """

    result = kg.query(
        query,
        params={
            "name": name,
            "description": description
        }
    )
    
    return result

In [195]:
def create_relationship(parent_name: str, child_name: str, relation: str):
    query = f"""
    MATCH (p {{name:$parent_name}})
    MATCH (c {{name:$child_name}})
    MERGE (p)-[r:{relation}]->(c)
    RETURN p.name AS parent,
           type(r) AS relation,
           c.name AS child
    """

    result = kg.query(
        query,
        params={
            "parent_name": parent_name,
            "child_name": child_name
        }
    )
    
    return result
    

In [196]:
def create_fmea_nodes(file_name: str):    
    with open(file_name, mode="r") as file:
        reader = csv.DictReader(file)
        for row in reader:
            node_name = row["Name"]
            node_description = row["Description"]
            node_type = row["Type"]
            create_node(node_type, node_name, node_description)

In [197]:
def build_fmea_tree(file_name: str):    
    with open(file_name, mode="r") as file:
        reader = csv.DictReader(file)
        for row in reader:
            node1 = row["Primary Node"]
            relation_type = row["Relation Type"]
            node2 = row["Secondary Node"]
            create_relationship(node1, node2, relation_type)

In [198]:
delete_existing_data()

In [199]:
create_fmea_nodes("./fmea_nodes.csv")

In [200]:
build_fmea_tree("./fmea_nodes_relations.csv")

In [201]:
# Select embeddings model.
embeddings = OpenAIEmbeddings(
    api_key=OPENAI_API_KEY,
    model="text-embedding-3-small"
)

In [202]:
# Create text for embedding of each node in the graph database.
def create_embedding():
    query = """
    MATCH (n)
    RETURN elementId(n) as nodeId,
        labels(n) as labels,
        n.name as name,
        n.description as description
    """
    nodes = kg.query(query)

    # Generate vector embeddings for each node and update the graph database with the embedding vector.
    for node in nodes:

        text = f"""
        Type: {node['labels'][0]}
        Name: {node['name']}
        Description: {node['description']}
        """

        vector = embeddings.embed_query(text)

        kg.query("""
            MATCH (n)
            WHERE elementId(n)=$nodeId
            SET n.embedding=$embedding
        """,
        params={
            "nodeId": node["nodeId"],
            "embedding": vector
        })

In [203]:
def delete_existing_vector_indexes():
    kg.query("""
    DROP INDEX fmea_vector_index IF EXISTS
    """)

In [204]:
def show_vector_indexes():
    print(kg.query("""
    SHOW VECTOR INDEXES
    """))

In [205]:
# Create Neo4j Vector Index
def create_vector_index():
    kg.query("""
    CREATE VECTOR INDEX fmea_vector_index IF NOT EXISTS
    FOR (n:Entity)
    ON (n.embedding)
    OPTIONS {
        indexConfig: {
            `vector.dimensions`: 1536,
            `vector.similarity_function`: 'cosine'
        }
    }
    """)

In [206]:
delete_existing_vector_indexes()
show_vector_indexes()

[]


In [207]:
create_embedding()
create_vector_index()
show_vector_indexes()

[{'id': 5, 'name': 'fmea_vector_index', 'state': 'ONLINE', 'populationPercent': 100.0, 'type': 'VECTOR', 'entityType': 'NODE', 'labelsOrTypes': ['Entity'], 'properties': ['embedding'], 'indexProvider': 'vector-3.0', 'owningConstraint': None, 'lastRead': None, 'readCount': None}]


In [220]:
def talk(question: str):
    
    query_vector = embeddings.embed_query(question)
    results = kg.query("""
    CALL db.index.vector.queryNodes(
        'fmea_vector_index',
        5,
        $embedding
    )
    YIELD node, score

    RETURN node.name as name,
        node.description as description,
        labels(node) as labels,
        score
    """,
    params={
        "embedding": query_vector
    })
    
    graph_context = ""
    if results:
        matched_node = results[0]["name"]
        print(f"Matched Node: {matched_node}")
        graph_context = kg.query("""
        MATCH (n {name:$name})-[r*1..2]->(m)
        RETURN
            labels(n) as NodeType,
            n.name AS Node,
            [rel IN r | type(rel)] AS Relations,
            m.name AS AssociatedNode
        """,
        params={
            "name": matched_node
        })
        
    context = ""
    for row in graph_context:
        context += str(row) + "\n"
   
    prompt = f"""
    You are an FMEA (Failure Modes and Effects Analysis) expert.

    Refer the graph context to answer the question related to FMEA.
    Kindly provide straight forward answer.
     
    Exampple:
    Question: Which could be the potential child system element of Car Air filtration System?
    Answer: The potential child system element of Car Air filtration System could be the "Air Filter Housing" or "Filter Element".
     
    Context:
    {context}

    Question:
    {question}
    """
    
    response = chat.invoke(prompt)
    
    return response.content

In [224]:
question = "Which could be the potential child system element of Body edge scrubbing?"
response = talk(question)
print(response)

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated procedure. ('db.index.vector.queryNodes' has been replaced by 'SEARCH')} {position: line: 2, column: 5, offset: 5} for query: "\n    CALL db.index.vector.queryNodes(\n        'fmea_vector_index',\n        5,\n        $embedding\n    )\n    YIELD node, score\n\n    RETURN node.name as name,\n        node.description as description,\n        labels(node) as labels,\n        score\n    "


Matched Node: Body edge scrubbing process
Answer: The potential child system elements of Body edge scrubbing could be the "Charge Air connector pipe operation," "Pre cleaner Body Clamp," or "Maintain internal profile."


In [222]:
question = "Which could be the potential Characteristics of 'HVAC' function element?"
response = talk(question)
print(response)

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated procedure. ('db.index.vector.queryNodes' has been replaced by 'SEARCH')} {position: line: 2, column: 5, offset: 5} for query: "\n    CALL db.index.vector.queryNodes(\n        'fmea_vector_index',\n        5,\n        $embedding\n    )\n    YIELD node, score\n\n    RETURN node.name as name,\n        node.description as description,\n        labels(node) as labels,\n        score\n    "


Matched Node: Maintain airflow through HVAC system
Answer: The potential characteristics of the 'HVAC' function element could be "Inspect-Dimension_10" and "Inspect-Dimension_11_10_7".
